In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt


In [3]:
data = pd.read_csv('data/cleaned_data.csv')
data.head()

,product_name,brands,categories_en,labels_en,ingredients_text,allergens_en,additives_en,nutrition_grade_fr,energy_100g,fat_100g,...,cocoa_100g,carbon-footprint_100g,nutrition-score-fr_100g,nutrition-score-uk_100g,category_level_1,category_level_2,category_level_3,category_level_4,category_level_5,category_level_6
0,Banana Chips Sweetened (Whole),not mentioned,NaN,Labels are missing,"Bananas, vegetable oil (coconut oil, corn oil ...",unknown,No additives,d,2243.0,28.57,...,0.0,no information,14.0,14.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Peanuts,torn & glasser,NaN,Labels are missing,"Peanuts, wheat flour, sugar, rice flour, tapio...","en:wheat, en:soy, en:peanuts",No additives,b,1941.0,17.86,...,0.0,no information,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Organic Salted Nut Mix,grizzlies,NaN,Labels are missing,"Organic hazelnuts, organic cashews, organic wa...",unknown,No additives,d,2540.0,57.14,...,0.0,no information,12.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Organic Polenta,bob's red mill,NaN,Labels are missing,Organic polenta,unknown,No additives,not given,1552.0,1.43,...,0.0,no information,not given,not given,NaN,NaN,NaN,NaN,NaN,NaN
4,Breadshop Honey Gone Nuts Granola,unfi,NaN,Labels are missing,"Rolled oats, grape concentrate, expeller press...",en:sesame,No additives,not given,1933.0,18.27,...,0.0,no information,not given,not given,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
us_data = pd.read_csv('data/us_data.csv')
us_data.shape

(171521, 98)

In [5]:
from modules.ingredients import clean_ingredients

data['ingredients'] = data['ingredients_text'].apply(clean_ingredients)
us_data['ingredients'] = us_data['ingredients_text'].apply(clean_ingredients)

In [6]:
data = data.drop(columns=['ingredients_text'])
us_data = us_data.drop(columns=['ingredients_text'])

### Pre-processing 

In [6]:
# Clean and standardize text columns
def clean_text(text):
    if isinstance(text, str):
        return text.lower().strip()
    return text

data['product_name'] = data['product_name'].apply(clean_text)
data['ingredients'] = data['ingredients'].apply(clean_text)
data['allergens_en'] = data['allergens_en'].apply(clean_text)
data['category_level_1'] = data['category_level_1'].apply(clean_text)
data['category_level_2'] = data['category_level_2'].apply(clean_text)

In [7]:
# Replace placeholders with NaN or empty lists
data['allergens_en'] = data['allergens_en'].replace('unknown', np.nan)
data['ingredients'] = data['ingredients'].replace('ingredients are missing', np.nan)

In [8]:
# Split columns into lists
data['ingredients'] = data['ingredients'].str.split(', ')
data['allergens_en'] = data['allergens_en'].str.split(', ')

### Implement product matching

In [29]:
from fuzzywuzzy import process

# Function to find top 5 closest matches
def find_top_matches(user_input, choices, limit=5):
    matches = process.extract(user_input, choices, limit=limit)
    return matches

# Example: User inputs a product name
user_input = "Sea Salt Potato Chips"
top_matches = find_top_matches(user_input, data['product_name'].tolist())

print(f"Top matches for '{user_input}':")
for match, score in top_matches:
    print(f"- {match} (Score: {score})")

Top matches for 'Sea Salt Potato Chips':
- sea salt potato chips (Score: 100)
- sea salt potato chips (Score: 100)
- organic sea salt potato chips (Score: 95)
- sweet potato chips sea salt (Score: 95)
- potato chips, sea salt (Score: 95)


### Category filtering

In [24]:
def get_primary_category(categories_en):
    if pd.isna(categories_en):
        return None
    # Split the hierarchy by commas and get the last part
    categories = categories_en.split(',')
    return categories[-1].strip()  # Return the last category (primary category)

def find_primary_category_from_matches(top_matches, df):
    for match, score in top_matches:
        categories_en = df[df['product_name'] == match]['categories_en'].values[0]
        primary_category = get_primary_category(categories_en)
        if primary_category is not None:
            return primary_category
    return 'product categories not found'

### Allergens filtering

In [25]:
def filter_by_allergens(products, allergens_to_avoid):
    for allergen in allergens_to_avoid:
        if f'contains_{allergen}' in products.columns:
            products = products[~products[f'contains_{allergen}']]
    return products

### Recommendations

In [26]:
def generate_recommendations(filtered_products, top_n=5):
    return filtered_products[['product_name', 'additives_en']].head(top_n)

In [27]:
def recommend_products(user_input, allergens_to_avoid, df, top_n=5):
    # Step 1: Find top 5 closest matches
    top_matches = find_top_matches(user_input, df['product_name'].tolist())
    
    # Step 2: Find primary category from top matches
    primary_category = find_primary_category_from_matches(top_matches, df)
    
    if primary_category == 'product categories not found':
        return 'product categories not found'
    
    # Step 3: Filter products in the primary category
    same_category_products = df[df['categories_en'].str.contains(primary_category, case=False, na=False)]
    
    # Step 4: Filter by allergens
    filtered_products = filter_by_allergens(same_category_products, allergens_to_avoid)
    
    # Step 5: Generate recommendations
    recommendations = generate_recommendations(filtered_products, top_n)
    
    return recommendations

In [28]:
# Example usage
user_input = "Sea Salt Potato Chips"  # User's input product name
allergens_to_avoid = ['soy', 'peanuts']  # Allergens to avoid
recommendations = recommend_products(user_input, allergens_to_avoid, us_data)

# Print recommendations
print("Final recommendations:")
print(recommendations)

Final recommendations:
                                             product_name  additives_en
232                                 Sea Salt Potato Chips  No additives
9505   Tims extra thick and crunchy jalapeno potato chips          E621
9517                   Original Kettle Style Potato Chips  No additives
18635                         Original Sweet Potato Chips  No additives
22732                                              Fritos  No additives


In [23]:
pd.set_option('display.max_colwidth', None)
us_data[~us_data.categories_en.isna()][['product_name', 'categories_en', 'category_level_1', 'category_level_2', 'category_level_3', 'category_level_4', 'category_level_5']].head(10)

,product_name,categories_en,category_level_1,category_level_2,category_level_3,category_level_4,category_level_5
232,Sea Salt Potato Chips,"Chips and fries,Chips",Chips and fries,Chips,NaN,NaN,NaN
358,mostly mesquite honey,"Spreads,Breakfasts,Sweet spreads,Bee products,Farming products,Sweeteners,Honeys",Spreads,Breakfasts,Sweet spreads,Bee products,Farming products
362,Clam Chowder A Condensed Soup,"Meals,Soups,Chowders",Meals,Soups,Chowders,NaN,NaN
386,Pizza Parlanno,"Meals,Pizzas pies and quiches,Pizzas,Pizzas-et-tartes-surgelees,Pizzas-surgelees,Pizzas-tartes-salees-et-quiches,Plats-prepares,Plats-prepares-surgeles,Surgeles",Meals,Pizzas pies and quiches,Pizzas,Pizzas-et-tartes-surgelees,Pizzas-surgelees
410,Mac 'n Cheese,"Meals,Microwave meals",Meals,Microwave meals,NaN,NaN,NaN
441,Vanilla Nonfat Yogurt,"Dairies,Yogurts,Low-fat yogurts",Dairies,Yogurts,Low-fat yogurts,NaN,NaN
487,Tortellini,"Plant-based foods and beverages,Plant-based foods,Cereals and potatoes,Cereals and their products,Pastas",Plant-based foods and beverages,Plant-based foods,Cereals and potatoes,Cereals and their products,Pastas
515,Hello Panda,"Sugary snacks,Biscuits and cakes,Biscuits,Cookies,Snacks",Sugary snacks,Biscuits and cakes,Biscuits,Cookies,Snacks
671,Spaghetti sauce with mushrooms,"Groceries,Sauces,Pasta sauces",Groceries,Sauces,Pasta sauces,NaN,NaN
678,100% desert mesquite honey,"Parve,U",Parve,U,NaN,NaN,NaN


### Radial chart

In [11]:
# Group by category_level_1 and calculate the mean of nutritional columns
macro_nutrition = ['energy_100g', 'proteins_100g', 'carbohydrates_100g', 'fiber_100g', 'fat_100g']
category_nutrition = us_data.groupby('category_level_1')[macro_nutrition].mean().reset_index()

In [12]:
# Get the top 10 categories by product count
top_categories = us_data['category_level_1'].value_counts().nlargest(10).index
top_category_nutrition = category_nutrition[category_nutrition['category_level_1'].isin(top_categories)]

In [13]:
from sklearn.preprocessing import MinMaxScaler

# Normalize the nutritional columns
scaler = MinMaxScaler()
top_category_nutrition[macro_nutrition] = scaler.fit_transform(top_category_nutrition[macro_nutrition])

/var/folders/fm/08v38d894qqg0x2sfcmb_2mh0000gn/T/ipykernel_82154/2469817990.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_category_nutrition[macro_nutrition] = scaler.fit_transform(top_category_nutrition[macro_nutrition])


In [17]:
import plotly.express as px
import plotly.graph_objects as go

# Function to create an interactive radar chart
def create_interactive_radar_chart(categories, values, title):
    fig = go.Figure()

    for i, category in enumerate(categories):
        fig.add_trace(go.Scatterpolar(
            r=values.iloc[i].tolist(),  # Nutritional values
            theta=values.columns,       # Nutritional metrics
            fill='toself',              # Fill the area under the line
            name=category               # Category name
        ))

    # Update layout for better visualization
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1]  # Normalized scale
            )
        ),
        title=title,
        showlegend=True
    )

    # Show the chart
    fig.show()

# Prepare data for the radar chart
categories = top_category_nutrition['category_level_1']
values = top_category_nutrition[macro_nutrition]

# Create the interactive radar chart
create_interactive_radar_chart(categories, values, title='Top 10 Primary Categories by Nutritional Facts')

In [7]:
us_data.columns.values

array(['product_name', 'brands', 'categories_en', 'labels_en',
       'allergens_en', 'additives_en', 'nutrition_grade_fr',
       'energy_100g', 'fat_100g', 'saturated-fat_100g',
       '-caprylic-acid_100g', '-capric-acid_100g', '-lauric-acid_100g',
       '-myristic-acid_100g', '-palmitic-acid_100g', '-stearic-acid_100g',
       '-arachidic-acid_100g', '-montanic-acid_100g',
       'monounsaturated-fat_100g', 'polyunsaturated-fat_100g',
       'omega-3-fat_100g', '-alpha-linolenic-acid_100g',
       '-eicosapentaenoic-acid_100g', '-docosahexaenoic-acid_100g',
       'omega-6-fat_100g', '-linoleic-acid_100g',
       '-arachidonic-acid_100g', '-gamma-linolenic-acid_100g',
       'omega-9-fat_100g', '-oleic-acid_100g', 'trans-fat_100g',
       'cholesterol_100g', 'carbohydrates_100g', 'sugars_100g',
       '-sucrose_100g', '-glucose_100g', '-fructose_100g',
       '-lactose_100g', '-maltose_100g', '-maltodextrins_100g',
       'starch_100g', 'polyols_100g', 'fiber_100g', 'proteins_100g